# TRUEGUARD — Faithful External Baseline Comparison (`SelfCheckGPT-NLI` & `Semantic Entropy`)
Evaluates scientifically faithful implementations of:
1. **SelfCheckGPT-NLI** (Manakul et al., EMNLP 2023) using the official `potsawee/deberta-v3-large-mnli` checkpoint with $M = 10$ stochastic samples (`T=0.7, top_p=0.9`).
2. **Semantic Entropy** (Kuhn et al., ICLR 2023; Farquhar et al., Nature 2024) using `microsoft/deberta-large-mnli` bidirectional entailment clustering + length-normalized sequence log-likelihoods (and discrete SE) over the same $M = 10$ samples.

### Strict Fairness & Zero-Leakage Guarantees
- Uses the **exact 250 calibration and 250 test TriviaQA questions, original TinyLlama greedy answers, and ground-truth labels** saved in `/content/drive/MyDrive/TRUEGUARD_IEEE_FINAL/`.
- Frozen `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (0 trainable weights).
- Operating thresholds for F1 are selected **strictly on the 250 calibration split** (zero test-label access).
- Measures **complete end-to-end GPU latency** (sampling + NLI scoring) and peak VRAM, and saves all outputs under `/content/drive/MyDrive/TRUEGUARD_IEEE_FINAL/external_validation/` without overwriting any original result.


In [1]:
# STEP 1: Dependencies, Google Drive Mount & Directory Setup
!pip install -q "transformers>=4.44.0" "accelerate>=0.30.0" "huggingface_hub>=0.24.0" "sentencepiece" "protobuf" "scikit-learn>=1.3.0" "scipy>=1.11.0" "matplotlib>=3.7.0" "tabulate>=0.9.0" "pandas>=2.0.0" "pyarrow>=14.0.0"

import os, sys, json, time, math, random, gc
import numpy as np
import pandas as pd
import torch
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/TRUEGUARD_IEEE_FINAL"
EXT_DIR  = os.path.join(BASE_DIR, "external_validation")
os.makedirs(EXT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | GPU: {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'}")
assert DEVICE == "cuda", "Please select T4 GPU in Runtime -> Change runtime type."


Mounted at /content/drive
Device: cuda | GPU: Tesla T4


In [2]:
# STEP 2: Load Exact Saved TriviaQA Calibration (250) & Test (250) Splits + Original Greedy Answers
from huggingface_hub import hf_hub_download

# Reconstruct exact question order (same seed=42, offset=0/250 as TRUEGUARD_IEEE_Experiments_FINAL.ipynb)
QA_INSTR = "Answer the question briefly in one short sentence.\nQuestion: "
pq_path = hf_hub_download("mandarjoshi/trivia_qa", "rc.nocontext/validation-00000-of-00001.parquet", repo_type="dataset")
df_tqa = pd.read_parquet(pq_path)

def get_tqa_questions(n, seed=42, offset=0):
    idx = np.random.RandomState(seed).permutation(len(df_tqa))[offset:offset + n]
    items = []
    for i in idx:
        r = df_tqa.iloc[int(i)]
        aliases = [a for a in list(r["answer"]["normalized_aliases"]) if a]
        if aliases:
            items.append(dict(id=int(i), question=str(r["question"]).strip(),
                              user_text=QA_INSTR + str(r["question"]).strip(), gold=aliases))
    return items

CAL_ITEMS  = get_tqa_questions(250, seed=42, offset=0)
TEST_ITEMS = get_tqa_questions(250, seed=42, offset=250)

cal_ck = np.load(os.path.join(BASE_DIR, "checkpoints", "triviaqa_cal_250.npz"), allow_pickle=True)
te_ck  = np.load(os.path.join(BASE_DIR, "checkpoints", "triviaqa_test_250.npz"), allow_pickle=True)
te_pr  = np.load(os.path.join(BASE_DIR, "predictions", "triviaqa_test_predictions.npz"))
lat_tg = json.load(open(os.path.join(BASE_DIR, "results", "real_latency.json"), "r"))

cal_kept = [int(i) for i in cal_ck["kept_idx"]]
te_kept  = [int(i) for i in te_ck["kept_idx"]]
CAL_QS   = [CAL_ITEMS[i]  for i in cal_kept]
TEST_QS  = [TEST_ITEMS[i] for i in te_kept]

CAL_ANS  = [str(a) for a in cal_ck["ans"]]
TEST_ANS = [str(a) for a in te_ck["ans"]]
Y_CAL    = cal_ck["y"].astype(int)
Y_TEST   = te_ck["y"].astype(int)
assert np.array_equal(Y_TEST, te_pr["y"].astype(int)), "Label alignment mismatch!"
print(f"Loaded exact TriviaQA splits: cal={len(Y_CAL)} (halluc={Y_CAL.mean():.3f}), test={len(Y_TEST)} (halluc={Y_TEST.mean():.3f})")
print(f"Sample test Q0: {TEST_QS[0]['question']} | Greedy ans: {TEST_ANS[0]!r} | Label: {Y_TEST[0]}")


rc.nocontext/validation-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 7.34MB            

rc.nocontext/validation-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Loaded exact TriviaQA splits: cal=250 (halluc=0.692), test=250 (halluc=0.660)
Sample test Q0: Farthings were in use in England as coinage from the 13th century and ceased to be legal tender on 31 December 1960. How many of them were there in a pound (1)? | Greedy ans: 'Farthings were in use in England as coinage from the 13th century until 1960' | Label: 1


In [3]:
# STEP 3: Generate M=10 Stochastic Samples + Sequence Log-Probabilities from Frozen TinyLlama-1.1B
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
M_SAMPLES  = 10
TEMP       = 0.7
TOP_P      = 0.9
MAX_NEW    = 24

SAMPLES_CKPT = os.path.join(EXT_DIR, f"triviaqa_m{M_SAMPLES}_samples.json")

if os.path.exists(SAMPLES_CKPT):
    print(f"[CACHE] Loading previously generated M={M_SAMPLES} samples from {SAMPLES_CKPT}")
    SAMP_DATA = json.load(open(SAMPLES_CKPT, "r", encoding="utf-8"))
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)
    model.eval()
    for p in model.parameters(): p.requires_grad = False
    torch.cuda.reset_peak_memory_stats()

    @torch.no_grad()
    def sample_m_with_logprobs(user_text):
        prompt = tok.apply_chat_template([{"role": "user", "content": user_text}], tokenize=False, add_generation_prompt=True)
        ids = tok(prompt, return_tensors="pt", add_special_tokens=True).input_ids[:, -384:].to(DEVICE)
        plen = ids.shape[1]
        # Batched generation of M_SAMPLES stochastic continuations
        ids_rep = ids.repeat(M_SAMPLES, 1)
        out = model.generate(
            ids_rep, max_new_tokens=MAX_NEW, do_sample=True, temperature=TEMP, top_p=TOP_P,
            pad_token_id=tok.pad_token_id, attention_mask=torch.ones_like(ids_rep),
            return_dict_in_generate=True, output_scores=True
        )
        # Compute length-normalized sequence log-probability for each generated answer
        gen_ids = out.sequences[:, plen:]  # (M, T_gen)
        step_logps = torch.stack([torch.log_softmax(sc.float(), dim=-1) for sc in out.scores], dim=1)  # (M, T_gen, V)
        samples, mean_logps = [], []
        for m in range(M_SAMPLES):
            text_full = tok.decode(gen_ids[m], skip_special_tokens=True)
            ans_line = text_full.strip().split("\n")[0].strip()
            # Keep tokens up to EOS or newline
            toks = gen_ids[m]
            valid_len = 0
            lp_sum = 0.0
            for t in range(len(toks)):
                tid = int(toks[t])
                if tid == tok.eos_token_id:
                    break
                lp_sum += float(step_logps[m, t, tid])
                valid_len += 1
            samples.append(ans_line if ans_line else text_full.strip())
            mean_logps.append(lp_sum / max(valid_len, 1))
        return samples, mean_logps

    SAMP_DATA = {"config": dict(model=MODEL_NAME, M=M_SAMPLES, temperature=TEMP, top_p=TOP_P, max_new_tokens=MAX_NEW, seed=SEED),
                 "cal": [], "test": [], "timing": {}}
    for split_name, qs in [("cal", CAL_QS), ("test", TEST_QS)]:
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for i, q in enumerate(qs):
            s_list, lp_list = sample_m_with_logprobs(q["user_text"])
            SAMP_DATA[split_name].append({"samples": s_list, "mean_logps": lp_list})
            if (i + 1) % 50 == 0 or (i + 1) == len(qs):
                print(f"  [TinyLlama M={M_SAMPLES} {split_name}] {i+1}/{len(qs)} ({time.perf_counter()-t0:.1f}s)", flush=True)
        torch.cuda.synchronize()
        SAMP_DATA["timing"][f"tinyllama_sampling_{split_name}_sec"] = round(time.perf_counter() - t0, 3)

    SAMP_DATA["timing"]["tinyllama_peak_vram_gb"] = round(torch.cuda.max_memory_allocated() / (1024**3), 3)
    json.dump(SAMP_DATA, open(SAMPLES_CKPT, "w", encoding="utf-8"), indent=2)
    print(f"Saved {SAMPLES_CKPT} | Timing: {SAMP_DATA['timing']}")
    del model; gc.collect(); torch.cuda.empty_cache()


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

... (21 outputs left out)

[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [TinyLlama M=10 test] 200/250 (130.6s)


[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [TinyLlama M=10 test] 250/250 (163.6s)
Saved /content/drive/MyDrive/TRUEGUARD_IEEE_FINAL/external_validation/triviaqa_m10_samples.json | Timing: {'tinyllama_sampling_cal_sec': 172.514, 'tinyllama_sampling_test_sec': 163.623, 'tinyllama_peak_vram_gb': 2.177}


In [5]:
# STEP 4: Faithful SelfCheckGPT-NLI (Manakul et al., EMNLP 2023)
# Uses official potsawee/deberta-v3-large-mnli; scores P(contradiction | greedy_ans, sample_m) averaged over M=10.
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SC_MODEL_NAME = "potsawee/deberta-v3-large-mnli"
SC_CKPT = os.path.join(EXT_DIR, "selfcheckgpt_nli_scores.json")

if os.path.exists(SC_CKPT):
    SC_OUT = json.load(open(SC_CKPT, "r"))
    print(f"[CACHE] Loaded SelfCheckGPT-NLI scores from {SC_CKPT}")
else:
    print(f"Loading official SelfCheckGPT-NLI model: {SC_MODEL_NAME}...")
    sc_tok = AutoTokenizer.from_pretrained(SC_MODEL_NAME)
    sc_mod = AutoModelForSequenceClassification.from_pretrained(SC_MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
    sc_mod.eval()
    torch.cuda.reset_peak_memory_stats()

    # Verify label mapping in config (potsawee/deberta-v3-large-mnli: 0=entailment, 1=contradiction)
    id2lbl = {int(k): v.lower() for k, v in sc_mod.config.id2label.items()}
    print("SelfCheckGPT-NLI label map:", id2lbl)
    idx_ent = 0  # potsawee/deberta-v3-large-mnli: LABEL_0 = entailment
    idx_con = 1  # potsawee/deberta-v3-large-mnli: LABEL_1 = contradiction

    @torch.no_grad()
    def score_selfcheck_split(greedy_list, samp_list):
        scores = []
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for i, (g_ans, s_obj) in enumerate(zip(greedy_list, samp_list)):
            # Manakul et al. official SelfCheckNLI: pair = (hypothesis=sentence, premise=sampled_passage)
            # In potsawee/deberta-v3-large-mnli, tokenizer(sentence, sample_passage) -> [entailment, contradiction]
            samps = s_obj["samples"]
            enc = sc_tok([g_ans] * len(samps), samps, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
            logits = sc_mod(**enc).logits.float()
            # Softmax over [entailment, contradiction] ONLY (exact Manakul et al. Eq. 4)
            pair_logits = torch.stack([logits[:, idx_ent], logits[:, idx_con]], dim=-1)
            p_contra = torch.softmax(pair_logits, dim=-1)[:, 1].cpu().numpy()
            scores.append(float(np.mean(p_contra)))
            if (i + 1) % 50 == 0 or (i + 1) == len(greedy_list):
                print(f"  [SelfCheckGPT-NLI] {i+1}/{len(greedy_list)} ({time.perf_counter()-t0:.1f}s)", flush=True)
        torch.cuda.synchronize()
        return scores, round(time.perf_counter() - t0, 3)

    cal_sc, t_cal = score_selfcheck_split(CAL_ANS,  SAMP_DATA["cal"])
    te_sc,  t_te  = score_selfcheck_split(TEST_ANS, SAMP_DATA["test"])
    SC_OUT = dict(model=SC_MODEL_NAME, M=M_SAMPLES, cal_scores=cal_sc, test_scores=te_sc,
                  cal_nli_sec=t_cal, test_nli_sec=t_te,
                  nli_peak_vram_gb=round(torch.cuda.max_memory_allocated() / (1024**3), 3))
    json.dump(SC_OUT, open(SC_CKPT, "w"), indent=2)
    print(f"Saved {SC_CKPT} | test_nli_sec={t_te}s | peak_vram={SC_OUT['nli_peak_vram_gb']}GB")
    del sc_mod; gc.collect(); torch.cuda.empty_cache()


Loading official SelfCheckGPT-NLI model: potsawee/deberta-v3-large-mnli...


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

SelfCheckGPT-NLI label map: {0: 'label_0', 1: 'label_1'}
  [SelfCheckGPT-NLI] 50/250 (3.7s)
  [SelfCheckGPT-NLI] 100/250 (6.4s)
  [SelfCheckGPT-NLI] 150/250 (9.1s)
  [SelfCheckGPT-NLI] 200/250 (11.7s)
  [SelfCheckGPT-NLI] 250/250 (14.3s)
  [SelfCheckGPT-NLI] 50/250 (2.6s)
  [SelfCheckGPT-NLI] 100/250 (5.2s)
  [SelfCheckGPT-NLI] 150/250 (7.9s)
  [SelfCheckGPT-NLI] 200/250 (10.5s)
  [SelfCheckGPT-NLI] 250/250 (13.1s)
Saved /content/drive/MyDrive/TRUEGUARD_IEEE_FINAL/external_validation/selfcheckgpt_nli_scores.json | test_nli_sec=13.081s | peak_vram=1.004GB


In [7]:
# STEP 5: Faithful Semantic Entropy (Kuhn et al., ICLR 2023; Farquhar et al., Nature 2024)
# Uses official microsoft/deberta-large-mnli bidirectional entailment clustering + length-normalized sequence log-likelihoods.
SE_MODEL_NAME = "microsoft/deberta-large-mnli"
SE_CKPT = os.path.join(EXT_DIR, "semantic_entropy_scores.json")

if os.path.exists(SE_CKPT):
    SE_OUT = json.load(open(SE_CKPT, "r"))
    print(f"[CACHE] Loaded Semantic Entropy scores from {SE_CKPT}")
else:
    print(f"Loading official Semantic Entropy NLI model: {SE_MODEL_NAME}...")
    se_tok = AutoTokenizer.from_pretrained(SE_MODEL_NAME)
    se_mod = AutoModelForSequenceClassification.from_pretrained(SE_MODEL_NAME).to(DEVICE)
    se_mod.eval()
    torch.cuda.reset_peak_memory_stats()

    id2lbl = {int(k): v.lower() for k, v in se_mod.config.id2label.items()}
    print("Semantic Entropy NLI label map:", id2lbl)
    idx_ent = [k for k, v in id2lbl.items() if "entail" in v][0]
    idx_con = [k for k, v in id2lbl.items() if "contra" in v][0]

    @torch.no_grad()
    def semantically_equivalent(q_text, a, b):
        if a.strip().lower() == b.strip().lower():
            return True
        t1 = f"{q_text} {a}"
        t2 = f"{q_text} {b}"
        enc = se_tok([t1, t2], [t2, t1], padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
        preds = se_mod(**enc).logits.argmax(dim=-1).cpu().tolist()
        # Strict Kuhn et al. / Farquhar et al. bidirectional equivalence: neither contradicts and at least one entails
        return (idx_con not in preds) and (idx_ent in preds)

    def score_se_split(qs, samp_list):
        se_ln_scores, se_disc_scores, n_clusters_list = [], [], []
        torch.cuda.synchronize(); t0 = time.perf_counter()
        for i, (q, s_obj) in enumerate(zip(qs, samp_list)):
            samps = s_obj["samples"]
            logps = np.array(s_obj["mean_logps"], dtype=np.float64)
            # Greedy semantic clustering (exact Kuhn et al. Algorithm 1)
            clusters = []  # list of list of indices
            for m in range(len(samps)):
                placed = False
                for cl in clusters:
                    if semantically_equivalent(q["question"], samps[m], samps[cl[0]]):
                        cl.append(m); placed = True; break
                if not placed:
                    clusters.append([m])
            n_clusters_list.append(len(clusters))
            # 1. Discrete Semantic Entropy (Farquhar et al., Nature 2024)
            probs_disc = np.array([len(cl) / len(samps) for cl in clusters], dtype=np.float64)
            se_disc = float(-np.sum(probs_disc * np.log(probs_disc + 1e-12)))
            # 2. Length-Normalized Sequence-Log-Likelihood Semantic Entropy (Kuhn et al., ICLR 2023)
            max_lp = float(np.max(logps))
            weights = np.exp(logps - max_lp)
            cl_weights = np.array([weights[cl].sum() for cl in clusters], dtype=np.float64)
            probs_ln = cl_weights / np.sum(cl_weights)
            se_ln = float(-np.sum(probs_ln * np.log(probs_ln + 1e-12)))
            se_ln_scores.append(se_ln); se_disc_scores.append(se_disc)
            if (i + 1) % 50 == 0 or (i + 1) == len(qs):
                print(f"  [Semantic Entropy] {i+1}/{len(qs)} ({time.perf_counter()-t0:.1f}s)", flush=True)
        torch.cuda.synchronize()
        return se_ln_scores, se_disc_scores, n_clusters_list, round(time.perf_counter() - t0, 3)

    cal_ln, cal_dc, _, t_cal = score_se_split(CAL_QS,  SAMP_DATA["cal"])
    te_ln,  te_dc,  te_nc, t_te = score_se_split(TEST_QS, SAMP_DATA["test"])
    SE_OUT = dict(model=SE_MODEL_NAME, M=M_SAMPLES,
                  cal_se_ln=cal_ln, test_se_ln=te_ln,
                  cal_se_disc=cal_dc, test_se_disc=te_dc,
                  test_mean_clusters=round(float(np.mean(te_nc)), 3),
                  cal_nli_sec=t_cal, test_nli_sec=t_te,
                  nli_peak_vram_gb=round(torch.cuda.max_memory_allocated() / (1024**3), 3))
    json.dump(SE_OUT, open(SE_CKPT, "w"), indent=2)
    print(f"Saved {SE_CKPT} | test_nli_sec={t_te}s | mean_clusters={SE_OUT['test_mean_clusters']} | peak_vram={SE_OUT['nli_peak_vram_gb']}GB")
    del se_mod; gc.collect(); torch.cuda.empty_cache()


Loading official Semantic Entropy NLI model: microsoft/deberta-large-mnli...


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Semantic Entropy NLI label map: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}
  [Semantic Entropy] 50/250 (55.9s)
  [Semantic Entropy] 100/250 (105.1s)
  [Semantic Entropy] 150/250 (160.5s)
  [Semantic Entropy] 200/250 (211.9s)
  [Semantic Entropy] 250/250 (254.1s)
  [Semantic Entropy] 50/250 (54.5s)
  [Semantic Entropy] 100/250 (98.9s)
  [Semantic Entropy] 150/250 (151.7s)
  [Semantic Entropy] 200/250 (198.5s)
  [Semantic Entropy] 250/250 (251.9s)
Saved /content/drive/MyDrive/TRUEGUARD_IEEE_FINAL/external_validation/semantic_entropy_scores.json | test_nli_sec=251.939s | mean_clusters=4.036 | peak_vram=2.474GB


In [8]:
# STEP 6: Unified Head-to-Head Comparison, Bootstrap CIs, Holm-Bonferroni Tests, Latency & Figures
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from tabulate import tabulate
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Fit ISP-only cleanly on the exact same 250 calibration split (matching phase5)
F_cal, H_cal = cal_ck["F"].astype(np.float64), cal_ck["H"].astype(np.float32)
F_te,  H_te  = te_ck["F"].astype(np.float64),  te_ck["H"].astype(np.float32)
best_isp = (-1, None)
for C in (0.001, 0.01, 0.1):
    oof = np.zeros((len(Y_CAL), 4))
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=42).split(F_cal, Y_CAL):
        for k in range(4):
            m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=300)).fit(H_cal[tr, k], Y_CAL[tr])
            oof[va, k] = m.predict_proba(H_cal[va, k])[:, 1]
    auc = roc_auc_score(Y_CAL, oof.mean(1))
    if auc > best_isp[0]: best_isp = (auc, C)
isp_probes = [make_pipeline(StandardScaler(), LogisticRegression(C=best_isp[1], max_iter=300)).fit(H_cal[:, k], Y_CAL) for k in range(4)]
isp_cal_s  = np.mean([p.predict_proba(H_cal[:, k])[:, 1] for k, p in enumerate(isp_probes)], axis=0)
isp_te_s   = np.mean([p.predict_proba(H_te[:, k])[:, 1]  for k, p in enumerate(isp_probes)], axis=0)

# Calibrate Youden thresholds STRICTLY on the 250 calibration split (zero test-label access!)
def youden_thr(y, s):
    fpr, tpr, thr = roc_curve(y, s)
    return float(thr[np.argmax(tpr - fpr)])

def aurc(correct, risk):
    order = np.argsort(risk, kind="stable")
    err = 1.0 - correct[order]
    return float((np.cumsum(err) / np.arange(1, len(err) + 1)).mean())

def acc_at_cov(correct, risk, cov):
    order = np.argsort(risk, kind="stable")
    k = max(1, int(round(cov * len(order))))
    return float(correct[order[:k]].mean())

def holm(pvals):
    m = len(pvals); order = np.argsort(pvals); adj = np.zeros(m); rm = 0.0
    for r, idx in enumerate(order):
        rm = max(rm, min(1.0, (m - r) * pvals[idx])); adj[idx] = rm
    return adj.tolist()

mit_rep = json.load(open(os.path.join(BASE_DIR, "results", "mitigation_report.json"), "r"))
TW_TG   = float(mit_rep["tau_warning"])

METHODS = {
    "TRUEGUARD_4sensor":      (None,                         te_pr["risk_calibrated"],    TW_TG),
    "TRUEGUARD_ISP_only":     (isp_cal_s,                    isp_te_s,                    youden_thr(Y_CAL, isp_cal_s)),
    "SelfCheckGPT_NLI":       (np.array(SC_OUT["cal_scores"]), np.array(SC_OUT["test_scores"]), youden_thr(Y_CAL, SC_OUT["cal_scores"])),
    "Semantic_Entropy_LN":    (np.array(SE_OUT["cal_se_ln"]),  np.array(SE_OUT["test_se_ln"]),  youden_thr(Y_CAL, SE_OUT["cal_se_ln"])),
    "Semantic_Entropy_Disc":  (np.array(SE_OUT["cal_se_disc"]),np.array(SE_OUT["test_se_disc"]),youden_thr(Y_CAL, SE_OUT["cal_se_disc"])),
    "Predictive_Entropy":     (F_cal[:, 6],                  F_te[:, 6],                  youden_thr(Y_CAL, F_cal[:, 6])),
    "NLL":                    (F_cal[:, 8],                  F_te[:, 8],                  youden_thr(Y_CAL, F_cal[:, 8])),
}

gc_test = 1 - Y_TEST
rng = np.random.RandomState(42)
N_BOOT = 1000
b_idx = [rng.choice(len(Y_TEST), len(Y_TEST), replace=True) for _ in range(N_BOOT)]

SUMMARY = {}
boot_auc, boot_aurc, boot_acc46 = {}, {}, {}
for name, (_, sc, thr) in METHODS.items():
    sc = np.asarray(sc, dtype=np.float64)
    ba  = np.array([roc_auc_score(Y_TEST[i], sc[i]) if len(np.unique(Y_TEST[i])) == 2 else np.nan for i in b_idx])
    ba  = ba[~np.isnan(ba)]
    br  = np.array([aurc(gc_test[i], sc[i]) for i in b_idx])
    b46 = np.array([acc_at_cov(gc_test[i], sc[i], 0.46) for i in b_idx])
    boot_auc[name] = ba; boot_aurc[name] = br; boot_acc46[name] = b46
    pred = (sc >= thr).astype(int)
    SUMMARY[name] = dict(
        threshold_from_cal=round(float(thr), 4),
        auroc=round(float(roc_auc_score(Y_TEST, sc)), 4),
        auroc_ci95=[round(float(np.percentile(ba, 2.5)), 4), round(float(np.percentile(ba, 97.5)), 4)],
        auprc=round(float(average_precision_score(Y_TEST, sc)), 4),
        f1_at_cal_thr=round(float(f1_score(Y_TEST, pred, zero_division=0)), 4),
        aurc=round(aurc(gc_test, sc), 4),
        aurc_ci95=[round(float(np.percentile(br, 2.5)), 4), round(float(np.percentile(br, 97.5)), 4)],
        acc_at_46pct=round(acc_at_cov(gc_test, sc, 0.46), 4),
        acc_at_50pct=round(acc_at_cov(gc_test, sc, 0.50), 4),
        risk_coverage={f"{c}%": round(acc_at_cov(gc_test, sc, c / 100.0), 4) for c in (100, 90, 80, 70, 60, 50, 46)},
    )

# Paired bootstrap tests: TRUEGUARD_4sensor vs every other method on AURC & AUROC
PAIRED = {}
p_list_aurc, keys_comp = [], []
for name in METHODS:
    if name == "TRUEGUARD_4sensor": continue
    da = boot_auc["TRUEGUARD_4sensor"] - boot_auc[name]
    dr = boot_aurc["TRUEGUARD_4sensor"] - boot_aurc[name]
    p_auc  = float(2 * min((da <= 0).mean(), (da >= 0).mean()))  # two-sided bootstrap p
    p_aurc = float(2 * min((dr >= 0).mean(), (dr <= 0).mean()))
    PAIRED[f"TG_vs_{name}"] = dict(
        delta_auroc=round(SUMMARY["TRUEGUARD_4sensor"]["auroc"] - SUMMARY[name]["auroc"], 4),
        p_auroc_2sided=round(p_auc, 4),
        delta_aurc=round(SUMMARY["TRUEGUARD_4sensor"]["aurc"] - SUMMARY[name]["aurc"], 4),
        p_aurc_2sided=round(p_aurc, 4),
    )
    p_list_aurc.append(p_aurc); keys_comp.append(f"TG_vs_{name}")
for k, pa in zip(keys_comp, holm(p_list_aurc)):
    PAIRED[k]["p_aurc_holm"] = round(float(pa), 4)

# End-to-end per-question GPU latency comparison (seconds per question for 250 test questions)
t_samp_per_q = SAMP_DATA["timing"]["tinyllama_sampling_test_sec"] / len(Y_TEST)
t_sc_nli_q   = SC_OUT["test_nli_sec"] / len(Y_TEST)
t_se_nli_q   = SE_OUT["test_nli_sec"] / len(Y_TEST)
t_tg_post_q  = (lat_tg["trueguard_postpass_ms_per_token"] * 24) / 1000.0
t_tg_on_q    = (lat_tg["trueguard_online_ms_per_token"] * 24) / 1000.0

COST_TABLE = {
    "TRUEGUARD_postpass": dict(generations_per_q=1, extra_models=0, sec_per_q=round(t_tg_post_q, 4), speedup_vs_selfcheck=round((t_samp_per_q + t_sc_nli_q) / t_tg_post_q, 2), speedup_vs_se=round((t_samp_per_q + t_se_nli_q) / t_tg_post_q, 2)),
    "TRUEGUARD_online":   dict(generations_per_q=1, extra_models=0, sec_per_q=round(t_tg_on_q, 4),   speedup_vs_selfcheck=round((t_samp_per_q + t_sc_nli_q) / t_tg_on_q, 2),   speedup_vs_se=round((t_samp_per_q + t_se_nli_q) / t_tg_on_q, 2)),
    "SelfCheckGPT_NLI":   dict(generations_per_q=1 + M_SAMPLES, extra_models="potsawee/deberta-v3-large-mnli (435M)", sampling_sec_per_q=round(t_samp_per_q, 4), nli_sec_per_q=round(t_sc_nli_q, 4), total_sec_per_q=round(t_samp_per_q + t_sc_nli_q, 4), nli_peak_vram_gb=SC_OUT["nli_peak_vram_gb"]),
    "Semantic_Entropy":   dict(generations_per_q=M_SAMPLES,     extra_models="microsoft/deberta-large-mnli (435M)",  sampling_sec_per_q=round(t_samp_per_q, 4), nli_sec_per_q=round(t_se_nli_q, 4), total_sec_per_q=round(t_samp_per_q + t_se_nli_q, 4), nli_peak_vram_gb=SE_OUT["nli_peak_vram_gb"]),
}

FINAL_EXT = dict(
    n_cal=int(len(Y_CAL)), n_test=int(len(Y_TEST)), M_samples=M_SAMPLES,
    deviations_documented=[
        "TriviaQA answers are single short phrases/sentences (max_new_tokens=24), so SelfCheckGPT-NLI evaluates the full answer as one sentence (identical to sentence-level Manakul et al.).",
        "Semantic Entropy uses greedy bidirectional entailment clustering with microsoft/deberta-large-mnli (Kuhn et al. Algorithm 1) over M=10 samples; both length-normalized log-likelihood SE and discrete SE are reported.",
        "Thresholds for F1 were calibrated strictly on the 250-question calibration split using Youden's J (zero test-label access).",
    ],
    metrics=SUMMARY, paired_bootstrap_tests=PAIRED, end_to_end_cost=COST_TABLE,
)
out_p = os.path.join(EXT_DIR, "external_baselines_report.json")
json.dump(FINAL_EXT, open(out_p, "w"), indent=2)
np.savez_compressed(os.path.join(EXT_DIR, "external_baseline_predictions.npz"),
                    y=Y_TEST, tg_4sensor=te_pr["risk_calibrated"], tg_isp=isp_te_s,
                    selfcheck_nli=np.array(SC_OUT["test_scores"]),
                    se_ln=np.array(SE_OUT["test_se_ln"]), se_disc=np.array(SE_OUT["test_se_disc"]),
                    entropy=F_te[:, 6], nll=F_te[:, 8])

# Print master comparison table
rows = [[n, f"{v['auroc']:.4f} [{v['auroc_ci95'][0]:.3f},{v['auroc_ci95'][1]:.3f}]", f"{v['auprc']:.4f}", f"{v['f1_at_cal_thr']:.4f}",
         f"{v['aurc']:.4f} [{v['aurc_ci95'][0]:.3f},{v['aurc_ci95'][1]:.3f}]", f"{v['acc_at_46pct']*100:.1f}%", f"{v['acc_at_50pct']*100:.1f}%"]
        for n, v in SUMMARY.items()]
print("\n=== EXTERNAL BASELINE COMPARISON ON IDENTICAL 250 TRIVIAQA TEST QUESTIONS ===")
print(tabulate(rows, headers=["Method", "AUROC [95% CI]", "AUPRC", "F1 (Cal Thr)", "AURC [95% CI]", "Acc@46%", "Acc@50%"], tablefmt="grid"))
print("\n=== PAIRED BOOTSTRAP TESTS (TRUEGUARD 4-SENSOR vs BASELINES) ===")
print(json.dumps(PAIRED, indent=2))
print("\n=== END-TO-END COMPUTATIONAL COST (SECONDS / QUESTION ON GPU) ===")
print(json.dumps(COST_TABLE, indent=2))

# Risk-coverage figure (300 DPI)
covs = [100, 90, 80, 70, 60, 50, 46]
plt.figure(figsize=(7.6, 4.8))
plot_spec = [
    ("TRUEGUARD_4sensor",   "TRUEGUARD 4-Sensor",       "#1f77b4", "-",  "o", 2.3),
    ("TRUEGUARD_ISP_only",  "TRUEGUARD ISP-Only",       "#ff7f0e", "--", "s", 1.7),
    ("SelfCheckGPT_NLI",    "SelfCheckGPT-NLI (M=10)",  "#d62728", "-",  "^", 2.0),
    ("Semantic_Entropy_LN", "Semantic Entropy (M=10)",  "#2ca02c", "-.", "D", 2.0),
    ("Predictive_Entropy",  "Predictive Entropy",       "#9467bd", ":",  "v", 1.5),
    ("NLL",                 "Length-Normalized NLL",    "#7f7f7f", ":",  "x", 1.4),
]
for k, lbl, col, ls, mk, lw in plot_spec:
    plt.plot(covs, [SUMMARY[k]["risk_coverage"][f"{c}%"] * 100 for c in covs], label=f"{lbl} (AURC={SUMMARY[k]['aurc']:.3f})", color=col, ls=ls, marker=mk, lw=lw, ms=5)
plt.gca().invert_xaxis()
plt.xlabel("Answer Coverage (%)", fontsize=10); plt.ylabel("Selective Accuracy Among Answered (%)", fontsize=10)
plt.title("TriviaQA (n=250 Test): TRUEGUARD vs. SelfCheckGPT-NLI & Semantic Entropy", fontsize=10.5, fontweight="bold")
plt.legend(fontsize=8, loc="upper left"); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(EXT_DIR, "fig_external_baselines_risk_coverage.png"), dpi=300); plt.close()
print(f"\nSaved {out_p} and fig_external_baselines_risk_coverage.png")



=== EXTERNAL BASELINE COMPARISON ON IDENTICAL 250 TRIVIAQA TEST QUESTIONS ===
+-----------------------+----------------------+---------+----------------+----------------------+-----------+-----------+
| Method                | AUROC [95% CI]       |   AUPRC |   F1 (Cal Thr) | AURC [95% CI]        | Acc@46%   | Acc@50%   |
+=======================+======================+=========+================+======================+===========+===========+
| TRUEGUARD_4sensor     | 0.7876 [0.720,0.852] |  0.846  |         0.8084 | 0.4360 [0.360,0.523] | 55.6%     | 54.4%     |
+-----------------------+----------------------+---------+----------------+----------------------+-----------+-----------+
| TRUEGUARD_ISP_only    | 0.7706 [0.701,0.834] |  0.8476 |         0.8062 | 0.4473 [0.369,0.532] | 53.9%     | 52.0%     |
+-----------------------+----------------------+---------+----------------+----------------------+-----------+-----------+
| SelfCheckGPT_NLI      | 0.7477 [0.681,0.812] |  0.8537 |  